# Acquisition des Donnees

Ce chapitre couvre l'ingestion du fichier brut `marketing_campaign.csv`, l'audit memoire et la premiere inspection qualite.

## Chargement du Dataset Brut

In [1]:
import os, sys, warnings
warnings.filterwarnings('ignore')

PROJ = r'c:/Users/ahoudzi/apti/aptispace-datascience-projet'
sys.path.insert(0, PROJ)
import pandas as pd
import numpy as np

RAW_PATH = os.path.join(PROJ, 'data', 'raw', 'marketing_campaign.csv')
df = pd.read_csv(RAW_PATH, sep='\t', engine='c')
print(f'Dimensions : {df.shape[0]} lignes x {df.shape[1]} colonnes')
print(f'Memoire    : {df.memory_usage(deep=True).sum()/1024:.1f} KB')

Dimensions : 2240 lignes x 29 colonnes
Memoire    : 561.4 KB


## Types et Valeurs Manquantes

In [2]:
info = pd.DataFrame({
    'dtype':  df.dtypes,
    'nulls':  df.isnull().sum(),
    'null_%': (df.isnull().sum()/len(df)*100).round(2),
    'unique': df.nunique()
})
print(info.to_string())

                       dtype  nulls  null_%  unique
ID                     int64      0    0.00    2240
Year_Birth             int64      0    0.00      59
Education                str      0    0.00       5
Marital_Status           str      0    0.00       8
Income               float64     24    1.07    1974
Kidhome                int64      0    0.00       3
Teenhome               int64      0    0.00       3
Dt_Customer              str      0    0.00     663
Recency                int64      0    0.00     100
MntWines               int64      0    0.00     776
MntFruits              int64      0    0.00     158
MntMeatProducts        int64      0    0.00     558
MntFishProducts        int64      0    0.00     182
MntSweetProducts       int64      0    0.00     177
MntGoldProds           int64      0    0.00     213
NumDealsPurchases      int64      0    0.00      15
NumWebPurchases        int64      0    0.00      15
NumCatalogPurchases    int64      0    0.00      14
NumStorePurc

## Downcasting - Optimisation Memoire

In [3]:
mem_avant = df.memory_usage(deep=True).sum()/1024
for col in df.select_dtypes(include=['int64']).columns:
    df[col] = pd.to_numeric(df[col], downcast='integer')
for col in df.select_dtypes(include=['float64']).columns:
    df[col] = pd.to_numeric(df[col], downcast='float')
mem_apres = df.memory_usage(deep=True).sum()/1024
gain = (1 - mem_apres/mem_avant)*100
print(f'Avant : {mem_avant:.1f} KB | Apres : {mem_apres:.1f} KB | Gain : -{gain:.0f}%')

Avant : 561.4 KB | Apres : 187.4 KB | Gain : -67%


## Audit Qualite - Anomalies Detectees

| Anomalie | Variable | Valeur | Action |
|----------|----------|--------|--------|
| Valeurs manquantes MNAR | Income | 24 NaN | Flag binaire + imputation mediane strate |
| Outliers physiques | Year_Birth | 1893, 1900 | Masquage (age > 100 ans) |
| Modalites non standards | Marital_Status | YOLO, Absurd | Harmonisation -> Other |
| Format date texte | Dt_Customer | DD-MM-YYYY | Conversion datetime64 + Customer_Days |

In [4]:
print('Valeurs manquantes Income :', df['Income'].isnull().sum())
print('Annees aberrantes :', df[df['Year_Birth'] < 1920]['Year_Birth'].tolist())
print('Marital_Status :', df['Marital_Status'].value_counts().to_dict())

Valeurs manquantes Income : 24
Annees aberrantes : [1900, 1893, 1899]
Marital_Status : {'Married': 864, 'Together': 580, 'Single': 480, 'Divorced': 232, 'Widow': 77, 'Alone': 3, 'Absurd': 2, 'YOLO': 2}


## Distribution de la Variable Cible

In [5]:
print(df['Response'].value_counts(normalize=True).round(3).to_string())
ratio = (df.Response==0).sum() / (df.Response==1).sum()
print(f'\nDesequilibre => scale_pos_weight recommande : {ratio:.1f}')

Response
0    0.851
1    0.149

Desequilibre => scale_pos_weight recommande : 5.7
